# Cervical Cancer Classifier — Kaggle Training Notebook

## What you need before running

| Requirement | How to set it up |
|---|---|
| **Herlev dataset** | Add Data → search `shubhrawat132/herlevdataset` — already attached at `/kaggle/input/herlevdataset` |
| **Internet ON** | Notebook Settings → Internet → ON  (needed to `git clone` the repo) |
| **GPU ON** | Notebook Settings → Accelerator → GPU T4 x2 |

No second dataset upload needed — the repo is cloned from GitHub in Cell 1.

## Flow
1. Clone repo from GitHub
2. Inspect dataset structure
3. Install requirements
4. Configure hyperparameters
5. Train
6. Zip & download checkpoints

In [ ]:
# ── Cell 1 — Find or clone the repo ───────────────────────────────────────────
# Try to find the repo in:
#   1. /kaggle/input/<dataset> (if uploaded as Kaggle Dataset)
#   2. Clone from GitHub if not found
#
# Requires: Internet ON to clone from GitHub

import os
import subprocess, sys
from pathlib import Path

DEFAULT_REPO_URL = 'https://github.com/Shubh-Rawat7/Cervical-Cancer-Classifier.git'
REPO_URL = os.environ.get('REPO_URL', DEFAULT_REPO_URL).strip()
# If you pushed your refactored code to a different GitHub repository,
# set the Kaggle environment variable REPO_URL or edit this default URL.
REPO_DIR = None

print(f'Using REPO_URL = {REPO_URL}')

# ── Try to find uploaded repo in Kaggle input ─────────────────────────────────
def find_uploaded_repo() -> Path | None:
    for candidate in Path('/kaggle/input').iterdir():
        if not candidate.is_dir():
            continue
        backend_dir = candidate / 'backend'
        if (backend_dir / 'train.py').exists():
            return candidate
    return None

REPO_DIR = find_uploaded_repo()
if REPO_DIR is not None:
    print(f'Found repo in uploaded dataset: {REPO_DIR}')
else:
    REPO_DIR = Path('/kaggle/working/repo')
    if REPO_DIR.exists():
        print('Repo already cloned — pulling latest changes...')
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull'])
    else:
        print('Cloning repo from GitHub...')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])

BACKEND_DIR = REPO_DIR / 'backend'

# ── Validate ──────────────────────────────────────────────────────────────────
if not BACKEND_DIR.exists():
    raise FileNotFoundError(
        f'backend/ not found under {REPO_DIR}\n'
        'Options:\n'
        '  1. Upload your project as a Kaggle Dataset: Add Data → Upload Data\n'
        '  2. Set REPO_URL to a GitHub repo that contains your refactored code.\n'
        '  3. Or clone from GitHub with Internet ON and a valid repo URL.\n'
    )

if not (BACKEND_DIR / 'train.py').exists():
    repo_files = '\n'.join(
        f'  {p.relative_to(REPO_DIR)}' for p in sorted(REPO_DIR.iterdir())
    )
    raise FileNotFoundError(
        f'backend/train.py not found under {REPO_DIR}\n'
        f'Files found at top-level:\n{repo_files}\n\n'
        'The default GitHub repo does not contain the refactored backend.\n'
        'Upload your project as a Kaggle Dataset or set REPO_URL to your repo URL.\n'
    )

print(f'\nRepo      : {REPO_DIR}')
print(f'Backend   : {BACKEND_DIR}')
print('\nFiles in backend/:')
for f in sorted(BACKEND_DIR.iterdir()):
    if f.is_file():
        print(f'  {f.name}')

In [ ]:
# ── Cell 2 — Inspect the Herlev dataset ──────────────────────────────────────
# Prints the folder structure so we can confirm the exact DATA_DIR path.

from pathlib import Path

CLASS_NAMES = ['Normal', 'CIN1', 'CIN2', 'CIN3', 'Cancer']

# Kaggle mounts datasets at /kaggle/input/<dataset-name>
# The slug shubhrawat132/herlevdataset → folder name is 'herlevdataset'
HERLEV_ROOT = Path('/kaggle/input/herlevdataset')

if not HERLEV_ROOT.exists():
    # Kaggle sometimes uses the full slug path
    HERLEV_ROOT = Path('/kaggle/input/datasets/shubhrawat132/herlevdataset')

print(f'Dataset root: {HERLEV_ROOT}')
print(f'Exists      : {HERLEV_ROOT.exists()}')
print()

# Show top-level structure
if HERLEV_ROOT.exists():
    for entry in sorted(HERLEV_ROOT.iterdir()):
        if entry.is_dir():
            n_files = len(list(entry.rglob('*.*')))
            print(f'  {entry.name}/  ({n_files} files)')
        else:
            print(f'  {entry.name}')
else:
    print('ERROR: Dataset not found!')
    print('Go to: Add Data → search shubhrawat132/herlevdataset → Add')
    print()
    print('All mounted datasets:')
    for p in sorted(Path('/kaggle/input').iterdir()):
        print(f'  {p}')

In [ ]:
# ── Cell 3 — Resolve DATA_DIR and install requirements ───────────────────────
# Finds whichever subfolder actually contains the class folders.

import subprocess, sys
from pathlib import Path

CLASS_NAMES = ['Normal', 'CIN1', 'CIN2', 'CIN3', 'Cancer']

# ── Resolve DATA_DIR ──────────────────────────────────────────────────────────
def find_data_dir(root: Path) -> Path | None:
    """Return the folder that directly contains ≥2 class subdirectories."""
    if not root.exists():
        return None
    # Direct match
    if sum((root / c).exists() for c in CLASS_NAMES) >= 2:
        return root
    # train/val/test split — return root so train.py can use --val-split
    for split in ('train', 'val', 'test'):
        sp = root / split
        if sp.exists() and sum((sp / c).exists() for c in CLASS_NAMES) >= 2:
            return root
    # Recursive search
    for candidate in root.rglob('*'):
        if candidate.is_dir() and sum((candidate / c).exists() for c in CLASS_NAMES) >= 2:
            return candidate
    return None

for root_candidate in [
    Path('/kaggle/input/herlevdataset'),
    Path('/kaggle/input/datasets/shubhrawat132/herlevdataset'),
]:
    DATA_DIR = find_data_dir(root_candidate)
    if DATA_DIR:
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        'Herlev dataset not found.\n'
        'Add it via: Add Data → shubhrawat132/herlevdataset'
    )

REPO_DIR    = Path('/kaggle/working/repo')
BACKEND_DIR = REPO_DIR / 'backend'
OUTPUT_DIR  = Path('/kaggle/working/checkpoints')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

found_classes = [c for c in CLASS_NAMES if (DATA_DIR / c).exists()]
print(f'Data dir : {DATA_DIR}')
print(f'Classes  : {found_classes}')
print(f'Backend  : {BACKEND_DIR}')
print(f'Output   : {OUTPUT_DIR}')

# Image counts per class
print('\nImage counts:')
for cls in CLASS_NAMES:
    cls_dir = DATA_DIR / cls
    if cls_dir.exists():
        count = len(list(cls_dir.glob('*.*')))
        print(f'  {cls:<10} {count}')

# ── Install requirements ──────────────────────────────────────────────────────
req = BACKEND_DIR / 'requirements.txt'
print(f'\nInstalling requirements from {req} ...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)])
print('Done.')

for path in (str(REPO_DIR), str(BACKEND_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)

In [ ]:
# ── Cell 4 — Hyperparameters ──────────────────────────────────────────────────
# Edit values here directly. Environment variables override if set.

import os
from pathlib import Path

# Paths (set by Cell 3 — must run Cell 3 first)
OUTPUT_DIR  = Path('/kaggle/working/checkpoints')
DATA_DIR    = DATA_DIR   # resolved in Cell 3

# ─── Edit these ───────────────────────────────────────────────────────────────
EPOCHS             = int(os.environ.get('EPOCHS',             '60'))
BATCH_SIZE         = int(os.environ.get('BATCH_SIZE',         '16'))   # reduce to 8 if OOM
IMAGE_SIZE         = int(os.environ.get('IMAGE_SIZE',         '224'))
VAL_SPLIT          = float(os.environ.get('VAL_SPLIT',        '0.2'))
LR                 = float(os.environ.get('LR',               '5e-5'))
WEIGHT_DECAY       = float(os.environ.get('WEIGHT_DECAY',     '1e-4'))
PATIENCE           = int(os.environ.get('PATIENCE',           '15'))
ACCUMULATION_STEPS = int(os.environ.get('ACCUMULATION_STEPS', '2'))    # increase to 4 if OOM
WORKERS            = int(os.environ.get('WORKERS',            '2'))
SEED               = int(os.environ.get('SEED',               '42'))
GAMMA              = float(os.environ.get('GAMMA',            '2.0'))
BETA               = float(os.environ.get('BETA',             '0.9999'))
LABEL_SMOOTHING    = float(os.environ.get('LABEL_SMOOTHING',  '0.05'))
DROPOUT            = float(os.environ.get('DROPOUT',          '0.2'))
ACTIVATION         = os.environ.get('ACTIVATION',             'silu')
UNDERSAMPLE        = os.environ.get('UNDERSAMPLE',            'random')
LOSS_TYPE          = os.environ.get('LOSS_TYPE',              'class_balanced_focal')
USE_KFOLD          = os.environ.get('USE_KFOLD',              '0') == '1'
K_FOLDS            = int(os.environ.get('K_FOLDS',            '5'))
# ──────────────────────────────────────────────────────────────────────────────

print(f'Data dir            : {DATA_DIR}')
print(f'Output dir          : {OUTPUT_DIR}')
print(f'Epochs              : {EPOCHS}')
print(f'Batch size          : {BATCH_SIZE}')
print(f'Image size          : {IMAGE_SIZE}')
print(f'Learning rate       : {LR}')
print(f'Validation split    : {VAL_SPLIT}')
print(f'Loss type           : {LOSS_TYPE}')
print(f'Undersample         : {UNDERSAMPLE}')
print(f'Patience            : {PATIENCE}')
print(f'Accumulation steps  : {ACCUMULATION_STEPS}')
print(f'Use k-fold          : {USE_KFOLD}')

In [ ]:
# ── Cell 5 — Check train.py arguments ────────────────────────────────────────
# Prints the help text of train.py so we know exactly which flags it accepts.
# If the flag list differs from what Cell 6 passes, fix Cell 6 accordingly.

import subprocess, sys
from pathlib import Path

train_script = Path('/kaggle/working/repo/backend/train.py')
result = subprocess.run(
    [sys.executable, str(train_script), '--help'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

In [ ]:
# ── Cell 6 — Run training ─────────────────────────────────────────────────────
# Streams output in real-time. Best checkpoint is saved to OUTPUT_DIR after
# every epoch so a kernel timeout won't lose your work.
#
# NOTE: If Cell 5 shows that train.py does NOT accept some of the flags below
# (e.g. --backbone, --loss-type), remove those lines from the cmd list.

import os, subprocess, sys
from pathlib import Path

BACKEND_DIR  = Path('/kaggle/working/repo/backend')
train_script = BACKEND_DIR / 'train.py'

if not train_script.exists():
    raise FileNotFoundError(f'{train_script} not found — did Cell 1 run successfully?')

cmd = [
    sys.executable, str(train_script),
    '--data-dir',           str(DATA_DIR),
    '--output-dir',         str(OUTPUT_DIR),
    '--epochs',             str(EPOCHS),
    '--batch-size',         str(BATCH_SIZE),
    '--image-size',         str(IMAGE_SIZE),
    '--val-split',          str(VAL_SPLIT),
    '--lr',                 str(LR),
    '--weight-decay',       str(WEIGHT_DECAY),
    '--patience',           str(PATIENCE),
    '--workers',            str(WORKERS),
    '--seed',               str(SEED),
]

# ── Optional flags — train.py may or may not support these ───────────────────
# Cell 5 tells you which flags exist. Comment out any that aren't listed.
OPTIONAL_FLAGS = {
    '--accumulation-steps': str(ACCUMULATION_STEPS),
    '--loss-type':          LOSS_TYPE,
    '--undersample':        UNDERSAMPLE,
    '--gamma':              str(GAMMA),
    '--beta':               str(BETA),
    '--label-smoothing':    str(LABEL_SMOOTHING),
    '--activation':         ACTIVATION,
    '--dropout':            str(DROPOUT),
}

# Auto-detect which optional flags train.py accepts
help_result = subprocess.run(
    [sys.executable, str(train_script), '--help'],
    capture_output=True, text=True
)
help_text = help_result.stdout + help_result.stderr
for flag, value in OPTIONAL_FLAGS.items():
    if flag in help_text:
        cmd.extend([flag, value])
    else:
        print(f'Skipping unsupported flag: {flag}')

if USE_KFOLD and '--use-kfold' in help_text:
    cmd.extend(['--use-kfold', '--k-folds', str(K_FOLDS)])

print('Running:')
print(' '.join(cmd))
print('─' * 70)

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
)
for line in proc.stdout:
    print(line, end='', flush=True)

exit_code = proc.wait()
if exit_code != 0:
    raise RuntimeError(f'Training failed with exit code {exit_code}')

print('\n' + '─' * 70)
print('Training complete!')

In [ ]:
# ── Cell 7 — List saved artifacts ────────────────────────────────────────────

from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/checkpoints')

artifacts = sorted(OUTPUT_DIR.rglob('*.*'))
if not artifacts:
    print('No artifacts found in', OUTPUT_DIR)
    print('Training may not have saved checkpoints yet.')
else:
    print(f'Artifacts in {OUTPUT_DIR}:')
    total = 0
    for a in artifacts:
        size = a.stat().st_size / 1e6
        total += size
        print(f'  {a.name:<45} {size:>8.1f} MB')
    print(f'  {"TOTAL":<45} {total:>8.1f} MB')

In [ ]:
# ── Cell 8 — Zip for download ─────────────────────────────────────────────────
# Download the zip from the Kaggle Output tab.
# Then in your local project root:
#   unzip herlev_checkpoints.zip
# Files land at: backend/Checkpoints/<filename>

import zipfile
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/checkpoints')
ZIP_PATH   = Path('/kaggle/working/herlev_checkpoints.zip')

CHECKPOINT_EXTS = {'.pt', '.pth', '.json', '.yaml', '.pkl', '.bin'}

artifacts = sorted(
    p for p in OUTPUT_DIR.rglob('*.*')
    if p.suffix in CHECKPOINT_EXTS
)

if not artifacts:
    raise FileNotFoundError(
        'No checkpoint files found. Make sure Cell 6 (training) completed successfully.'
    )

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for artifact in artifacts:
        arcname = str(Path('backend') / 'Checkpoints' / artifact.name)
        zf.write(artifact, arcname=arcname)
        print(f'  + {arcname}  ({artifact.stat().st_size/1e6:.1f} MB)')

print(f'\nZip size : {ZIP_PATH.stat().st_size / 1e6:.1f} MB')
print(f'Zip path : {ZIP_PATH}')
print()
print('Next steps:')
print('  1. Kaggle Output tab → Download herlev_checkpoints.zip')
print('  2. In your local project root:')
print('       unzip herlev_checkpoints.zip')
print('  3. Start the backend:')
print('       uvicorn backend.api.main:app --reload')

## Troubleshooting

| Problem | Cause | Fix |
|---|---|---|
| `git clone` fails | Internet is OFF | Notebook Settings → Internet → ON |
| `NameError: DATA_DIR` | Cells run out of order | Run All (Run → Run All Cells) |
| `FileNotFoundError: herlevdataset` | Dataset not attached | Add Data → `shubhrawat132/herlevdataset` |
| CUDA out of memory | Batch too large | Set `BATCH_SIZE = 8`, `ACCUMULATION_STEPS = 4` in Cell 4 |
| Flag not recognized error | train.py doesn't support that arg | Cell 5 shows which flags exist; Cell 6 auto-skips unsupported ones |
| Kernel timeout mid-run | Kaggle 9-hour limit | Best checkpoint already saved; run Cell 7 + 8 to zip it |